# 037 — Flujo supervisado y partición train-validation-test

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Resumen de la materia

**Aprendizaje supervisado:** dado $D = \{(x_i, y_i)\}$ muestreado i.i.d. de $P(X,Y)$,
buscar $f$ que minimice el riesgo esperado $R(f) = E[L(f(X),Y)]$. Solo podemos medir el
riesgo empírico $\hat{R}$ sobre la muestra; la brecha $R - \hat{R}$ es la generalización.

**Tres particiones, tres funciones:**

- **Train** ajusta parámetros.
- **Validation** compara candidatos/hiperparámetros — su métrica se vuelve optimista con cada elección.
- **Test** se mira UNA sola vez al final: es la única estimación honesta del riesgo real.

**Fuga de datos:** cualquier información de evaluación que llega al entrenamiento
(duplicados, preprocesado ajustado con todo el dataset, fuga temporal, columnas
consecuencia del target). **Baseline:** el modelo trivial (clase mayoritaria, media)
que da denominador a toda métrica.


### 🔁 Flujo que sigue el laboratorio

```text
1. Fijar métrica, split y semilla     4. Comparar candidatos en desarrollo
2. Sellar el test                     5. Elegir UN modelo final
3. Entrenar candidatos con train      6. Medir una vez en test y reportar límites
```

El laboratorio `ml` genera candidatos de umbral y selecciona el de mejor accuracy de
desarrollo. Nota la limitación declarada en su salida: usa el mismo conjunto para
ilustrar la selección — exactamente el sesgo optimista que esta clase enseña a evitar.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

**Ejercicio 1.** El contrato incluye `kind="ml"`, `evidence` y `limitations`. Con
`seed=37` selecciona umbral 3.05 con accuracy 1.00 — medida sobre el mismo conjunto usado
para elegir, como declaran las `limitations`: es una métrica de *desarrollo*, no una
estimación de riesgo real.

**Ejercicio 2.** Train = índices {0,2,4,6,7,8} → valores (0,L)(1,L)(3,L)(4,S)(5,S)(6,S).
Con `t=2`: predice S para 3,4,5,6 → falla (3,L): 5/6 ≈ 0.83. Con `t=4`: predice S para
4,5,6 → 6/6 = 1.00. Validación {1:(1,L), 5:(3,S)}: con `t=2`,
1 < 2 predice L ✓ y 3 ≥ 2 predice S ✓ → 2/2; con `t=4`, 3 < 4 predice L para el spam ✗ → 1/2.
El ranking se invierte entre train y validación (train prefería t=4, validación t=2):
con muestras pequeñas la selección es inestable. Elegido `t=2`, test {3:(2,L), 9:(7,S)}:
2 ≥ 2 predice S ✗, 7 ≥ 2 ✓ → accuracy de test 0.5, igual al baseline mayoritario.

**Ejercicio 3.** La media y la desviación usadas para normalizar contienen información de
las filas de test: es fuga por preprocesado. Orden correcto: split → `fit` del
normalizador con train → `transform` de val/test. La cifra honesta será igual o menor
(típicamente menor), porque el 0.97 incorpora información del test.

**Ejercicio 4.** Los dos resultados son idénticos salvo el campo `seed`: este runner usa
un dataset y unos candidatos FIJOS — la semilla se registra en el contrato (declara la
intención de reproducibilidad) pero no gobierna ningún muestreo. En un experimento real la
semilla controlaría el split y cualquier bootstrap/inicialización; entonces la métrica
cambia con la semilla, y reportar una sola realización confunde muestreo con desempeño
esperado: lo correcto es media ± dispersión sobre varias semillas.


In [ ]:
result = run_lab("ml", seed=37)
assert result["kind"] == "ml"
assert result["evidence"]
show(result)


In [ ]:
# Ejercicio 2 — split y selección a mano, verificado con código
valores   = [0, 1, 1, 2, 3, 3, 4, 5, 6, 7]
etiquetas = ["L", "L", "L", "L", "L", "S", "S", "S", "S", "S"]
idx_test  = {3, 9}
idx_val   = {1, 5}
idx_train = [i for i in range(10) if i not in idx_test | idx_val]

def accuracy_umbral(t, indices):
    pred = ["S" if valores[i] >= t else "L" for i in indices]
    real = [etiquetas[i] for i in indices]
    return sum(p == r for p, r in zip(pred, real)) / len(indices)

for t in (2, 4):
    print(f"t={t}  train={accuracy_umbral(t, idx_train):.2f}  val={accuracy_umbral(t, sorted(idx_val)):.2f}")
mejor_t = 2  # gana en validación
print(f"test con t={mejor_t}: {accuracy_umbral(mejor_t, sorted(idx_test)):.2f}  (baseline mayoritario: 0.50)")


In [ ]:
# Ejercicio 4 — qué controla (y qué no) la semilla en este runner
r_a = run_lab("ml", seed=37)
r_b = run_lab("ml", seed=137)
difieren = {k for k in r_a if r_a[k] != r_b[k]}
print("claves que difieren:", difieren)          # solo {'seed'}
print("selected idéntico:", r_a["result"]["selected"] == r_b["result"]["selected"])
# El dataset del runner es fijo: la semilla se registra pero no muestrea nada.
# En un experimento real (split/bootstrap aleatorios) la métrica SÍ variaría con
# la semilla y se reportaría media ± desviación sobre varias.


## Reflexión

1. El laboratorio selecciona el umbral con el mismo conjunto en el que mide accuracy y lo
   declara en `limitations`. ¿En qué dirección está sesgada la accuracy reportada (1.00) y
   qué partición adicional haría honesta la estimación?
2. Si normalizas los datos con la media de TODO el dataset antes del split, ¿qué tipo de
   fuga cometes y por qué el test deja de estimar el riesgo real aunque el modelo nunca
   haya visto esas filas?
3. Tu modelo logra accuracy 0.93 y la clase mayoritaria es el 92 % de los casos. ¿Qué
   baseline reportarías y qué conclusión honesta admite esa diferencia de 0.01?
